# Assignment 5: RAG Agent
This notebook demonstrates a RAG Agent that can answer multi-step questions using a different data source (NASA's Perseverance Rover Wikipedia page).

In [ ]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu beautifulsoup4 python-dotenv tiktoken

In [ ]:
import os
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

# Load environment variables (e.g., OPENAI_API_KEY from a .env file if available)
load_dotenv()

# If you don't have a .env file, uncomment the line below and add your key directly
# os.environ["OPENAI_API_KEY"] = "sk-your-api-key-here"

## 1. Load Data Source
We use the Wikipedia page for NASA's Perseverance Rover. We parse only the main content to avoid loading unnecessary navigation links.

In [ ]:
url = "https://en.wikipedia.org/wiki/Perseverance_(rover)"
loader = WebBaseLoader(
    web_paths=(url,),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            class_="mw-content-container"
        )
    )
)
docs = loader.load()
print(f"Loaded {len(docs)} document(s)")

## 2. Split Documents and Create Vector Store
We split the loaded documents into smaller chunks and embed them into an in-memory FAISS vector store.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} document chunks.")

vectorstore = FAISS.from_documents(splits, OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

## 3. Create Retriever Tool and Agent
We create a tool from the retriever and initialize an OpenAI-based agent.

In [ ]:
retriever_tool = create_retriever_tool(
    retriever,
    "perseverance_rover_search",
    "Search for information about the Mars Perseverance rover. You must use this tool to answer any questions about its mission, instruments, landing, or history."
)
tools = [retriever_tool]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an advanced AI assistant tasked with answering questions about the Mars Perseverance rover.
You have access to a retriever tool with information from its Wikipedia page.
For complex, multi-step questions, you should break down the questions and use the tool multiple times if necessary to find all the pieces of information before synthesizing your final answer.
Always cite the facts you found."""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 4. Test Multi-Step Questions
Let's test the agent. Because we set `verbose=True`, you'll see the agent's thought process (how it calls the tool, looks at the observation, and constructs a final answer).

In [ ]:
question_1 = "What is the name of the helicopter attached to Perseverance, and on what date did the rover land on Mars?"
response_1 = agent_executor.invoke({"input": question_1})
print("\nFinal Answer: \n", response_1["output"])

In [ ]:
question_2 = "What are the names of the seven primary scientific instruments on the rover? Briefly describe what MEDA and MOXIE do."
response_2 = agent_executor.invoke({"input": question_2})
print("\nFinal Answer: \n", response_2["output"])